In [ ]:
### Time Series Modelling - Holt-Winters forecast: smokeless consumer growth

import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score


# [Preparing series: smokeless consumers (millions), 2022–2025]


# Holt-Winters (double exponential smoothing — trend only, no seasonality at annual freq)
hw_model = ExponentialSmoothing(s_vals, trend='add', seasonal=None, initialization_method='estimated')
hw_fit   = hw_model.fit(optimized=True)

n_forecast = 10
hw_forecast = hw_fit.forecast(n_forecast)
f_years     = np.arange(s_years[-1]+1, s_years[-1]+n_forecast+1)


# Also fitting polynomial regression as comparison - Combined snippet
poly = PolynomialFeatures(degree=2)

X_train = poly.fit_transform((s_years - s_years[0]).reshape(-1, 1))
X_future = poly.transform((f_years - s_years[0]).reshape(-1, 1))

lr = LinearRegression().fit(X_train, s_vals)
poly_forecast = lr.predict(X_future)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(s_years, s_vals, 'o-', lw=2.5, label='Actual')
ax.plot(f_years, hw_forecast, '--o', alpha=0.7, label='Holt-Winters')
ax.plot(f_years, poly_forecast, '--s', alpha=0.7,
        label=f'Polynomial (R²={r2:.3f})')

ax.fill_between(
    f_years,
    hw_forecast * 0.85,
    hw_forecast * 1.15,
    alpha=0.12,
    label='±15% band'
)

ax.set(
    title='Smokeless Consumer Forecast',
    xlabel='Year',
    ylabel='Consumers (millions)'
)

ax.legend()
plt.tight_layout()
plt.show()

# [Marking BAT's stated targets if known]


# Plotting the figures
ax.set_title('Smokeless consumer forecast — Holt-Winters vs polynomial', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Consumers (millions)')
ax.legend()
plt.tight_layout()
plt.savefig('fig1_consumer_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

forecast_df = pd.DataFrame({
    'Year': f_years,
    'Forecast Consumers (M)': hw_forecast.round(1)
})

print(forecast_df)